# Key Vocabulary

| Term    | Meaning                                  |
|:--------|:-----------------------------------------|
|**Chunk** | A small, contiguous piece of text or data. |
|**Embedding**|A vector (list of numbers) representing text meaning|
|**Vector DataBase**|A database that stores and searches vectors by similarity|
|**retrieval**|Finding the most relevant chunks for a given query|
|**Context Injection**|Adding retrieval chunks into the LLM's prompt|
|**Grounding**|Forcing the LLm to answer based on provided facts|
|**Knowledge Base**|The collection of documents the system can retrieve from|

In [41]:
#CELL 1

!pip install sentence-transformers chromadb groq pandas -q

print("Package installed successfully")

#sentence-transformers     : Creates text embedding (vectors)
#chromadb

Package installed successfully


In [42]:
#CELL 2

import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer
from groq import Groq

print("All libraries imported successfully")
print("Ready to build RAG system")

All libraries imported successfully
Ready to build RAG system


In [43]:
#CELL 3

import os

GROQ_API_KEY="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"

#os.environ stores he key as an environment variable
#This makes it accessible o the Groq library when it needs o authenticate
os.environ['GROQ_API_KEY']=GROQ_API_KEY

#Initialize the GRoq client
#groq_client is our connection to the GROQ API service
groq_client = Groq(api_key=GROQ_API_KEY)

print("Groq API client initialized")
print("Note: if you see an authentication error later, double check you api key.")

Groq API client initialized
Note: if you see an authentication error later, double check you api key.


In [44]:
#CELL 4

#we load the same college_notes.csv dataset we used on day 7
#this is our knowledge base - the document our RAG system will retrieve from.

#pd.read_cv() : Read a CSV file and return a pandas dataframe

df = pd.read_csv('college_notes.csv')

#lets see what the dataset looks like
print("Shape of dataset:",df.shape)

print("\nColumn names:", df.columns.tolist())

Shape of dataset: (15, 4)

Column names: ['note_id', 'subject', 'topic', 'content']


In [45]:
#CELL 5

print("Subject in the dataset:")
#df['subject'] : selects only the 'subject' column from the dataframe
#.value.count() : counts how many times each unique value appears
print(df['subject'].value_counts())

print("\nSample of topics:")
#We display the topic column to understand what knowledge is available
print(df[['note_id','subject','topic']].to_string(index=False))
#.to_string(index=False)  : converts to string and hides row numbers

print("\n Length of content (number of characters) for each note:")
#df['content'].apply(len) : Applies the len() function to every row in the content column
#len() returns the number of characters in a string

df['content_length'] = df['content'].apply(len)
print(df[['topic','content_length']].to_string(index=False))

Subject in the dataset:
subject
Data Engineering      5
Machine Learning      5
Generative AI         3
Python Programming    2
Name: count, dtype: int64

Sample of topics:
note_id            subject                          topic
   N001   Data Engineering                  ETL Pipelines
   N002   Data Engineering                  SQL Databases
   N003   Data Engineering                  Data Cleaning
   N004   Data Engineering       APIs and Data Collection
   N005   Data Engineering           Big Data and PySpark
   N006   Machine Learning            Supervised Learning
   N007   Machine Learning               Model Evaluation
   N008   Machine Learning            Feature Engineering
   N009   Machine Learning                 Decision Trees
   N010   Machine Learning                  Random Forest
   N011      Generative AI          Large Language Models
   N012      Generative AI             Prompt Engineering
   N013      Generative AI Retrieval Augmented Generation
   N014 Python 

In [46]:
#CELL 6

#documents  :A list of text stirngs   - these are the actual center
#Each item is the full text content of one noe
documents = df['content'].tolist()


ids=[f"note_{row['note_id']}" for row in df.to_dict('records')]

#f"note_{row['note_id']}"  : An f-string that creates note_1,note_2, etc
#df.to_dict('records')  : converts dataframe

metadatas = [
    {"subject":row['subject'],"topic":row["topic"]}
    for row in df.to_dict('records')
]

#for each row we store the subject and topic as metadata
# this helps us to understand what topic was retrived when we et results

print(f"Total chunks prepared  : {len(documents)}")
print(f"First document ID      : {ids[0]}")
print(f"First metadata         : {metadatas[0]}")
print(f"First document content: {documents[0][:100]}...")
print(f"First document subject: {metadatas[0]['subject']}")
print(f"First document topic  : {metadatas[0]['topic']}")

Total chunks prepared  : 15
First document ID      : note_N001
First metadata         : {'subject': 'Data Engineering', 'topic': 'ETL Pipelines'}
First document content: ETL stands for Extract Transform Load. It is the process of collecting raw data from different sourc...
First document subject: Data Engineering
First document topic  : ETL Pipelines


In [47]:
#CELL 7

print("Loading embedding model...")
print("This may take 30-60 secons on first run - model is being downloaded")
print("(Subsequent run will be faster as the model is cached)")

#SentenceTransformer()  : Loads a pre-trained model
#all-MiniLM-L6-v2   :the model name - a fast, accurate sentence embedding model
#MiniLm = mini Language model (faster than large model)
#L6 =6 tranformer layers
#v2 = version 2 (improved accuracy)
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

print("\nEmbedding model loaded successfully.")
#Quick tezt : embed one sentence and check the vector shape
test_embedding = embedding_model.encode("This is a test sentence")

#.encode()  : converts a text string into a numpy array (vector of numbers)

print(f"Test embedding shape : {test_embedding.shape}")

Loading embedding model...
This may take 30-60 secons on first run - model is being downloaded
(Subsequent run will be faster as the model is cached)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Embedding model loaded successfully.
Test embedding shape : (384,)


In [48]:
#CELL 8

chroma_client = chromadb.Client()

collection = chroma_client.get_or_create_collection(name="college_notes_rag")

print("chromadb client created.")
print(f"Collection name : college_notes_rag")
print(f"Documents in collection so far : {collection.count()}")

chromadb client created.
Collection name : college_notes_rag
Documents in collection so far : 15


In [49]:
#CELL 9

print("Generating embeddings for all 15 notes")
print("This may take 15-30 seconds...")

embeddings = embedding_model.encode(documents, show_progress_bar=True)
print(f"\nEmbedding matrix shape : {embeddings.shape}")

embeddings_list = embeddings.tolist()
#.tolist() converts a numpy array to a nested python list

#Add all documents, their documents, IDs, and metadata to ChromaDB
collection.add(
    documents=documents,             #The actual text content for each row
    metadatas=metadatas,             #subject and topic info for wach note
    ids=ids,                         #unique string IDs like 'not_1', 'note-2'
    embeddings=embeddings_list       #the vector representation of each row
)

#collection.add() : inserts documents with their embeddings into the collection

print(f"\nDocuments successfully added to chromaDB.")
print(f"Total documents in collection: {collection.count()}")
#should show 15 now

Generating embeddings for all 15 notes
This may take 15-30 seconds...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Embedding matrix shape : (15, 384)

Documents successfully added to chromaDB.
Total documents in collection: 15


In [50]:
#CELL 10

def retrieve_relevant_chunks(question, top=3):
  """
  Given a user question, retrieve the most relevant document chunks from chromaDB.

  Parameters:
      question(str)   : the user's question as a text string
      top(int)      : how many top results to return (default)


  """
  question_embedding = embedding_model.encode(question).tolist()

  results = collection.query(
      query_embeddings=[question_embedding],
      n_results=top
  )
  return results

print("Retrieval function defined successfully")
print("Function: retrieve_relevant_chunks(question,top=3)")

Retrieval function defined successfully
Function: retrieve_relevant_chunks(question,top=3)


In [51]:
#CELL 11 :Test the retrieval function
#Let's test our retriever with a  sample question abd see which notes it finds

test_question = "What is ETL and how does it work in data engineering?"
print(f"Test Question: {test_question}")
print("="*60)

# Call the retrieval function to get results
results = retrieve_relevant_chunks(test_question, top=3)

#The results dictionary contains
#results['documents']  : A list of lists  - the actual text of retrieved chunks
#results['distances']
print("\nTop 3 Retrieved chunks :")
print("="*60)

for i, (doc, dist, meta) in enumerate(zip(
    results['documents'][0],
    results['distances'][0],
    results['metadatas'][0]
)):
   print(f"\nResult {i+1} : ")
   print(f"  Subject  : {meta['subject']}")
   print(f"  Topic    : {meta['topic']}")
   print(f"  Distance : {dist:.4f}")
   #Distance : lower value = more similar to our question
   #chromDB uses L@ distance by default (Euclidean distance)
   print(f"  Content  : {doc[:120]}...")
   #we show only the first 120 characters to keep output readable

Test Question: What is ETL and how does it work in data engineering?

Top 3 Retrieved chunks :

Result 1 : 
  Subject  : Data Engineering
  Topic    : ETL Pipelines
  Distance : 0.2269
  Content  : ETL stands for Extract Transform Load. It is the process of collecting raw data from different sources transforming it i...

Result 2 : 
  Subject  : Data Engineering
  Topic    : APIs and Data Collection
  Distance : 1.0690
  Content  : An API or Application Programming Interface allows two software applications to talk to each other. In data engineering ...

Result 3 : 
  Subject  : Python Programming
  Topic    : Data Visualization
  Distance : 1.3375
  Content  : Data visualization is the process of representing data as charts graphs and visual formats. Python libraries like Matplo...


#What is Context Injection?
#The RAG prompt templete
```
SYSTEM
you are a helpful academic assistant. Answer questions based on ONLY on the provided context.
If the answer is not in the ocntext, say "I dont't have enough information to answer this"
Do not use your general training knowledge. Only use the context provided

USER:
Context:
---
[Retrieved Document 1]
---
[Retrieved Document 2]
---
[Retrieved Document 3]
---

Question : [User's Question]
Answer

In [52]:
#CELL 12 : Build the context string from retrieved chunks

def build_context_from_result(results):
  context_parts = []  #an empty list to collect fromatted chunks

  #loop hrough each retrieved document and its metadata
  for i, (doc, meta) in enumerate(zip(
      results['documents'][0],      #The text context of retrieved documents
      results['metadatas'][0]       #The metadata (subject, topic) of retrieved documents
  )):
    # format each chunk as a labeled section
    chunk_text = f"[Source {i+1}: {meta['subject']}-{meta['topic']}]\n{doc}"
    #f-string combines : source number, subject, topic and the document text
    #Example : "[Source 1 : Data Engineering - ETL Pipelines ] \nETL stands for ..."
    context_parts.append(chunk_text)

#"\n\n---\n\n".join() : Combines all parts with a --- divided between each
  context_str = "\n\n---\n\n".join(context_parts)
  return context_str

#Test it with previous retrieval results
context=build_context_from_result(results)
print("="*60)
print(context[:500]+"...")
#We show only the first 500 characters to keep output manageable
print(f"\nTotal context length: {len(context)} characters")

[Source 1: Data Engineering-ETL Pipelines]
ETL stands for Extract Transform Load. It is the process of collecting raw data from different sources transforming it into a clean and structured format and loading it into a database or data warehouse for analysis.

---

[Source 2: Data Engineering-APIs and Data Collection]
An API or Application Programming Interface allows two software applications to talk to each other. In data engineering APIs are used to fetch data from external services like weat...

Total context length: 850 characters


In [53]:
#CELL 13

def generate_rag_answer(question,context):
  system_prompt = """You are a helpful assistant for engineering students.
  You will be given context retrieved from a college knowledge base, and a student's question

  RULES:
  1.Answer ONLY using the information provided in the ocntext below.
  2. If the answer is not found in the context, say exactly:
  "I don't have enough information in my knowledge base to answer this question."
  3.Do not use your general training Knowledge.
  4.Keep answers clear, accurate, and beginner-friendly.
  5.Mention which source the information came from when possible."""

  #USER prompt : the context + question formatted as the user message
  user_prompt = f"""Context from Knowledge Base:
{context}
---

Student's Question :{question}

Please answer the question based only on the context provided above."""
  #Call the groq API with the messages
  response =groq_client.chat.completions.create(
      model="llama-3.1-8b-instant",
      #model : the LLM to use -- same model from Day 6
      messages=[
          {"role":"system","content":system_prompt},
          #system messages  :Sets the LLM's behaviour and rules
          {"role":"user","content":user_prompt}
          #user message : the actual query with context injected
      ],
      temperature = 0.1,
      #temperature = 0.1 : very low randomness - we want factual, consistent answers
      #for RAG, low temperature is preferred so the LLM sticks to the context
      max_tokens = 500
      #max_tokens = 500 : maximum length of generated response
  )
  #extract the text answer from the APi response object
  answer = response.choices[0].message.content
  #response.choices : A list of response options(usually 1)
  #[0]              : take the first (and only) choice
  #.message.content : the actual text of the message

  return answer

print("RAG generation function defined ")

RAG generation function defined 


In [72]:
#CELL 14

#Now we combine ALL steps into one pipeline
#This is a complete RAG pipeline: Question->Retrieve ->inject

def ask_college_assistant(question, top_k=3, verbose=True):
  if verbose:
    print(f"Question: {question}")
    print("="*60)
    print("Step 1: Retrieving relevant documents...")

  #STEP 1 : RETRIEVE -Find relevant chunks from ChromaDB
  results = retrieve_relevant_chunks(question, top=top_k)

  if verbose:
    print(f"Retrieved {top_k} chunks from the knowledge base:")
    for i, (meta, dist) in enumerate(zip(results['metadatas'][0], results['distances'][0])):
      print(f"   {i+1}.  {meta['subject']}-{meta['topic']} (Distance: {dist:.4f})")
    print("\nStep 2:Building context string...")

  #STEP 2:INJECT - format retrieved chunks
  context = build_context_from_result(results)

  if verbose:
    print(f"Context Built ({len(context)} characters)")
    print("\nStep 3:Sending to LLM for answer generation...")

  #STEP 3 : GENERATE - Send context + question to Groq LLM
  answer = generate_rag_answer(question, context)

  if verbose:
    print("\n" + "=" * 60)
    print("ANSWER:")
    print("="*60)
    print(answer)
    print("="*60)

  return answer


print("Complete RAG pipeline function ready.")
print("Function: ask_college_assistant(question, top_k=3)")

Complete RAG pipeline function ready.
Function: ask_college_assistant(question, top_k=3)


In [61]:
#CELL 15 Test the full RAG pipeline - Question 1

question_1 = "What is ETL and What are its three main stages?"

answer_1 = ask_college_assistant(question_1, top_k=3, verbose=True)

#EXPECTED : The assistant should correctly explain Extract, Transform and Load
#The answer should be granted in the college_notes content


Question: What is ETL and What are its three main stages?
Step 1: Retrieving relevant documents...
Retrieved 3 chunks from the knowledge base:
   1.  Data Engineering-ETL Pipelines
   2.  Generative AI-Retrieval Augmented Generation
   3.  Generative AI-Prompt Engineering

Step 2:Building context string...
Context Built (924 characters)

Step 3:Sending to LLM for answer generation...

ANSWER:
ETL stands for Extract Transform Load. 

According to the context, the three main stages of ETL are:

1. Extract: This stage involves collecting raw data from different sources.
2. Transform: This stage involves transforming the raw data into a clean and structured format.
3. Load: This stage involves loading the transformed data into a database or data warehouse for analysis.

Source: [Source 1: Data Engineering-ETL Pipelines]


In [62]:
#CELL 16  Question 2

question_2 = "How do embeddings help in building search systems?"

answer_2 = ask_college_assistant(question_2, top_k=3, verbose=True)

#EXPECTED: The assistant should explain embedding

Question: How do embeddings help in building search systems?
Step 1: Retrieving relevant documents...
Retrieved 3 chunks from the knowledge base:
   1.  Generative AI-Retrieval Augmented Generation
   2.  Generative AI-Large Language Models
   3.  Machine Learning-Feature Engineering

Step 2:Building context string...
Context Built (904 characters)

Step 3:Sending to LLM for answer generation...

ANSWER:
I don't have enough information in my knowledge base to answer this question.


In [63]:
#CELL 17

question_3 = "What is teh population of Tokyo?"

print("Testing with an out-of-scope question (not in college notes):")
answer_3 = ask_college_assistant(question_3, top_k=3, verbose=True)

#EXPECTED: The assistant should respond with something like:
#"I don't have enough information in my knowledge base to answer this question."


Testing with an out-of-scope question (not in college notes):
Question: What is teh population of Tokyo?
Step 1: Retrieving relevant documents...
Retrieved 3 chunks from the knowledge base:
   1.  Generative AI-Large Language Models
   2.  Data Engineering-SQL Databases
   3.  Data Engineering-Data Cleaning

Step 2:Building context string...
Context Built (793 characters)

Step 3:Sending to LLM for answer generation...

ANSWER:
I don't have enough information in my knowledge base to answer this question.

The context provided does not contain any information about the population of Tokyo or any other geographical location. It covers topics related to Large Language Models, SQL databases, and data cleaning.


#Comparison Table

| Feature    |           Without RAG  |      With RAG         |
|:-----------|:-----------------------|:----------------------|
|**Knowledge Source** | LLM training data(fixed) | Your custom documents(updatable)|
|**Hallucination risk**|High|Low|
|**Can use private daa**|No|Yes|
|**Cities sources**|yes(Training date)|No(You add new doc anytime)|
|**Cost**| Cheaper(shorter prompts)|Slightly higher(longer prompts with context)|

In [65]:
#CELL 18

#chromaDB supports metadata filtering - retrieve only from a specific subject
#This is useful when you want domain-specific answers

def retrieve_by_subject(question,subject_filter,top_k=2):
  #embed the question
  question_embedding = embedding_model.encode(question).tolist()

  #Query with the where filter - only search documents with the matching subjects
  results = collection.query(
      query_embeddings=[question_embedding],
      n_results=top_k,
      where={"subject":subject_filter}
      #where  : A filter condition applied to the metadata
      #This tells chromaDB to ONLY consider documents where subject == subject_filter
  )
  return results

#Test :Ask a question but only retrieve from GenAi notes
print("Retrieving only from GenAI subject:")
print("="*60)

filtered_results = retrieve_by_subject(
    question="How do LLMs generate text?",
    subject_filter="GenAI",
    top_k=2
)

for i, (doc, meta) in enumerate(zip(
    filtered_results['documents'][0],
    filtered_results['metadatas'][0]
)):
  print(f"\nResult {i+1}: [{meta['subject']}]{meta['topic']}")
  print(f"  Content: {doc[:100]}...")


Retrieving only from GenAI subject:



#PART 13 PRACTICE QUESTIONS
---
#BEGINNER QUESTION
---

Q1. What is hallucination in the context of LLMs?

Q2. What does RAG stand for? What problem does it solve?

Q3. What us the role of a vector databse in the RAG pipeline?

---
#INTERMEDIATE QUESTIONS
---

Q4. What is the difference between the indexing phase and the Querying phase of RAG?

Q5. Why must you use the same embedding model for both documents and queries?

Q6. Why is low temperature (e.g.0.1) preferred for RAG-based LLM calls?


---
#Coding Questions
---
Q7. Modify the ```ask_college_assistant()``` function to also display the distance scores of retrieved chunks in the ouput

Q8.Change the system prompt in ```generate_rag_answer()``` to instruct the LLM to always respond in bullet points

Q9.Add a function that returns only the topic name of retrieved chunks without their full content

### Q7. Modify the `ask_college_assistant()` function to also display the distance scores of retrieved chunks in the output.

The `ask_college_assistant()` function, as defined in `CELL 14`, already includes the functionality to display the distance scores for retrieved chunks. The `verbose=True` parameter enables this output. Below is the relevant part of the code and a demonstration:

In [76]:
def ask_college_assistant_q7(question, top_k=3, verbose=True):
  if verbose:
    print(f"Question: {question}")
    print("="*60)
    print("Step 1: Retrieving relevant documents...")

  #STEP 1 : RETRIEVE -Find relevant chunks from ChromaDB
  results = retrieve_relevant_chunks(question, top=top_k)

  if verbose:
    print(f"Retrieved {top_k} chunks from the knowledge base:")
    for i, (meta, dist) in enumerate(zip(results['metadatas'][0], results['distances'][0])):
      # This line displays the distance scores
      print(f"   {i+1}.  {meta['subject']}-{meta['topic']} (Distance: {dist:.4f})")
    print("\nStep 2:Building context string...")

  #STEP 2:INJECT - format retrieved chunks
  context = build_context_from_result(results)

  if verbose:
    print(f"Context Built ({len(context)} characters)")
    print("\nStep 3:Sending to LLM for answer generation...")

  #STEP 3 : GENERATE - Send context + question to Groq LLM
  answer = generate_rag_answer(question, context)

  if verbose:
    print("\n" + "=" * 60)
    print("ANSWER:")
    print("="*60)
    print(answer)
    print("="*60)

  return answer

print("Modified RAG pipeline function for Q7 ready.")

Modified RAG pipeline function for Q7 ready.


In [77]:
# Demonstration for Q7
print("Demonstrating Q7 solution:")
question_q7 = "What is ETL?"
ask_college_assistant_q7(question_q7, top_k=3, verbose=True)

Demonstrating Q7 solution:
Question: What is ETL?
Step 1: Retrieving relevant documents...
Retrieved 3 chunks from the knowledge base:
   1.  Data Engineering-ETL Pipelines (Distance: 0.3298)
   2.  Data Engineering-APIs and Data Collection (Distance: 1.4877)
   3.  Generative AI-Retrieval Augmented Generation (Distance: 1.5185)

Step 2:Building context string...
Context Built (882 characters)

Step 3:Sending to LLM for answer generation...

ANSWER:
ETL stands for Extract Transform Load. It is the process of collecting raw data from different sources, transforming it into a clean and structured format, and loading it into a database or data warehouse for analysis. 

[Source: Data Engineering-ETL Pipelines]


'ETL stands for Extract Transform Load. It is the process of collecting raw data from different sources, transforming it into a clean and structured format, and loading it into a database or data warehouse for analysis. \n\n[Source: Data Engineering-ETL Pipelines]'

### Q8. Change the system prompt in `generate_rag_answer()` to instruct the LLM to always respond in bullet points.

To address this, we'll modify the `system_prompt` within the `generate_rag_answer()` function to include an explicit instruction for bullet points. Below is the modified function and a demonstration.

In [78]:
def generate_rag_answer_q8(question,context):
  system_prompt = """You are a helpful assistant for engineering students.
  You will be given context retrieved from a college knowledge base, and a student's question

  RULES:
  1.Answer ONLY using the information provided in the ocntext below.
  2. If the answer is not found in the context, say exactly:
  "I don't have enough information in my knowledge base to answer this question."
  3.Do not use your general training Knowledge.
  4.Keep answers clear, accurate, and beginner-friendly.
  5.Mention which source the information came from when possible.
  6. Always provide answers in bullet-point format.
  """

  user_prompt = f"""Context from Knowledge Base:
{context}
---

Student's Question :{question}

Please answer the question based only on the context provided above."""

  response =groq_client.chat.completions.create(
      model="llama-3.1-8b-instant",
      messages=[
          {"role":"system","content":system_prompt},
          {"role":"user","content":user_prompt}
      ],
      temperature = 0.1,
      max_tokens = 500
  )
  answer = response.choices[0].message.content
  return answer

print("Modified RAG generation function for Q8 ready.")

Modified RAG generation function for Q8 ready.


In [79]:
# Demonstration for Q8
print("Demonstrating Q8 solution:")
question_q8 = "Explain the main components of a RAG system."

# We'll use the original retrieve_relevant_chunks for context
retrieved_results = retrieve_relevant_chunks(question_q8, top=3)
context_q8 = build_context_from_result(retrieved_results)

answer_q8 = generate_rag_answer_q8(question_q8, context_q8)
print("\nAnswer (should be in bullet points):\n")
print(answer_q8)

Demonstrating Q8 solution:

Answer (should be in bullet points):

Based on the context provided, here are the main components of a RAG system:

* **Retrieval Component**: This component retrieves relevant documents from a knowledge base (Source 1: Generative AI-Retrieval Augmented Generation).
* **Generation Component**: This component generates an answer based on the retrieved documents (Source 1: Generative AI-Retrieval Augmented Generation).

Note: The context does not provide detailed information about the architecture or other components of a RAG system, but these two components are mentioned as part of the RAG technique.


### Q9. Add a function that returns only the topic names of retrieved chunks without their full content.

Here's a new function `get_retrieved_topics()` that retrieves relevant chunks and then extracts only the topic names from their metadata. A demonstration follows.

In [80]:
def get_retrieved_topics(question, top_k=3):
  """
  Given a user question, retrieve the most relevant document chunks from ChromaDB
  and return only their topic names.

  Parameters:
      question (str): The user's question as a text string.
      top_k (int): How many top results to consider.

  Returns:
      list: A list of topic names (strings).
  """
  # Retrieve relevant chunks using the existing function
  results = retrieve_relevant_chunks(question, top=top_k)

  # Extract topic names from metadatas
  topic_names = []
  if 'metadatas' in results and results['metadatas'] and results['metadatas'][0]:
    for meta in results['metadatas'][0]:
      if 'topic' in meta:
        topic_names.append(meta['topic'])

  return topic_names

print("Function `get_retrieved_topics` defined successfully.")

Function `get_retrieved_topics` defined successfully.


In [81]:
# Demonstration for Q9
print("Demonstrating Q9 solution:")
question_q9 = "What is prompt engineering?"
topics_q9 = get_retrieved_topics(question_q9, top_k=2)

print(f"\nFor the question: '{question_q9}', the retrieved topics are:")
for topic in topics_q9:
  print(f"- {topic}")

Demonstrating Q9 solution:

For the question: 'What is prompt engineering?', the retrieved topics are:
- Prompt Engineering
- Feature Engineering


#PART 14 MINI PROJECT  -college knowledge assistant

**Project Description**
Built a complete **College Assistant** that:
1. Loads the college_notes.csv knowledge base
2. Indexes all notes in ChromaDB with embeddings
3. Accepts a student question
4. Retrieves the top 3 revelent notes
5. Injects them as context into a Groq LLM Prompt
6. Returns a clear, grounded answer with source citations
7. Handles questions outside the knowledge base gracefully

# Beginner Questions

### Q1. What is hallucination in the context of LLMs?

Hallucination occurs when an LLM generates information that appears correct but is actually false, inaccurate, or unsupported by facts.

### Q2. What does RAG stand for? What problem does it solve?

RAG stands for Retrieval-Augmented Generation. It solves the problem of outdated knowledge and hallucinations by retrieving relevant information from external documents before generating an answer.

### Q3. What is the role of a vector database in the RAG pipeline?

A vector database stores document embeddings and performs similarity search to retrieve the most relevant documents for a user's query.

---

# Intermediate Questions

### Q4. What is the difference between the indexing phase and querying phase?

Indexing Phase:
Documents are processed, converted into embeddings, and stored in the vector database.

Querying Phase:
The user's question is converted into an embedding, relevant documents are retrieved, and the LLM generates an answer using the retrieved context.

### Q5. Why must you use the same embedding model for both documents and queries?

Using the same embedding model ensures that documents and queries are represented in the same vector space, allowing accurate similarity comparisons.

### Q6. Why is a low temperature (e.g., 0.1) preferred for RAG-based LLM calls?

A low temperature reduces randomness and helps the model generate more factual, consistent, and context-grounded answers based on the retrieved documents.

---

# Coding Questions

### Q7. Modify the ask_college_assistant() function to also display the distance scores of retrieved chunks in the output.

Retrieve the distance scores from the vector database results and display them along with each retrieved chunk to indicate how closely the chunk matches the user's query.

### Q8. Change the system prompt in generate_rag_answer() to instruct the LLM to always respond in bullet points.

Update the system prompt with an instruction such as:
"Always provide answers in bullet-point format."

### Q9. Add a function that returns only the topic names of retrieved chunks without their full content.

Create a function that retrieves relevant chunks and returns only the topic names from the metadata instead of returning the full document content.

In [75]:
# ==========================================================
# MINI PROJECT : COLLEGE KNOWLEDGE ASSISTANT
# ==========================================================

import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer
from groq import Groq

# ----------------------------------------------------------
# 1. Load Knowledge Base
# ----------------------------------------------------------

df = pd.read_csv("college_notes.csv")

print(f"Loaded {len(df)} notes")

# ----------------------------------------------------------
# 2. Initialize Embedding Model
# ----------------------------------------------------------

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# ----------------------------------------------------------
# 3. Create ChromaDB Collection
# ----------------------------------------------------------

client_db = chromadb.Client()

collection = client_db.get_or_create_collection(
    name="college_notes"
)

# ----------------------------------------------------------
# 4. Store Notes with Embeddings
# ----------------------------------------------------------

documents = []
metadatas = []
ids = []

for i, row in df.iterrows():

    text = str(row["content"])

    documents.append(text)

    metadatas.append({
        "subject": row["subject"],
        "topic": row["topic"]
    })

    ids.append(str(i))

embeddings = embedding_model.encode(documents).tolist()

collection.add(
    documents=documents,
    embeddings=embeddings,
    metadatas=metadatas,
    ids=ids
)

print("Knowledge Base Indexed Successfully!")

# ----------------------------------------------------------
# 5. Initialize Groq
# ----------------------------------------------------------

groq_client = Groq(
    api_key="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"
)

# ----------------------------------------------------------
# 6. RAG Function
# ----------------------------------------------------------

def ask_college_assistant(question, top_k=3):

    # Retrieve Relevant Notes
    query_embedding = embedding_model.encode(question).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )

    docs = results["documents"][0]
    metas = results["metadatas"][0]

    # Handle Out-of-Scope Questions
    if len(docs) == 0:
        return "Sorry, this information is not available in the college knowledge base."

    # Build Context
    context = "\n\n".join(docs)

    prompt = f"""
You are a College Knowledge Assistant.

Answer ONLY using the provided context.

Context:
{context}

Question:
{question}

If the answer is not present in the context,
say:
'Sorry, this information is not available in the college knowledge base.'
"""

    # Generate Answer
    response = groq_client.chat.completions.create(
    model="llama-3.1-8b-instant",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.3
    )

    answer = response.choices[0].message.content

    # Add Citations
    citations = "\n\nSources:\n"

    for meta in metas:
        citations += f"- {meta['subject']} : {meta['topic']}\n"

    return answer + citations

# ----------------------------------------------------------
# 7. Ask Question
# ----------------------------------------------------------

question = input("Ask a Question: ")

answer = ask_college_assistant(question)

print("\nAnswer:\n")
print(answer)

Loaded 15 notes


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Knowledge Base Indexed Successfully!
Ask a Question: How do ML models work

Answer:

Machine learning (ML) models work by learning from data. They are trained on labeled data, where the input features are paired with correct output labels. The model uses this data to learn patterns and relationships, and then makes predictions for new, unseen inputs. This process involves the model adjusting its parameters to minimize the difference between its predictions and the actual output labels.

Sources:
- Machine Learning : Model Evaluation
- Generative AI : Large Language Models
- Machine Learning : Supervised Learning

